# Electron g-2 Final Confirmation + Path-Length Weighting

Same top configs as before, but now with path-length weighting:
amp *= 1.0 / (step + 1) ** 1.5

This down-weights longer paths → reduces variance and often boosts suppression.

Top configs (from sweep3):
- 29 steps, phase=41, fbs=7.2
- 28 steps, phase=39, fbs=7.5
- 27 steps, phase=40, fbs=7.5

Edit N_REPEATS or N_paths_total if needed.

In [ ]:
import cupy as cp
import numpy as np
from tqdm import tqdm
import time
import pandas as pd

# ====================== SETTINGS ======================
N_paths_total = 1_000_000_000   # 1 billion — try 2e9 if VRAM allows
BATCH_SIZE    = 25_000_000      # lower to 10M if OOM
N_REPEATS     = 10              # repeats per config

C = 9.6e-7
alpha = 1 / 137.035999084

configs = [
    {'name': 'Best: 29/41/7.2', 'walk_length': 29, 'phase_scale': 41.0, 'fbs_power': 7.2},
    {'name': 'Strong: 28/39/7.5', 'walk_length': 28, 'phase_scale': 39.0, 'fbs_power': 7.5},
    {'name': 'Strong: 27/40/7.5', 'walk_length': 27, 'phase_scale': 40.0, 'fbs_power': 7.5},
]

n_batches = (N_paths_total + BATCH_SIZE - 1) // BATCH_SIZE
print(f"Total paths per run: {N_paths_total:,} | Repeats per config: {N_REPEATS}")
print(f"Total runs: {len(configs) * N_REPEATS}")

In [ ]:
# ====================== LOAD FIXED ASSETS ======================
phi = (1 + np.sqrt(5)) / 2
inv_phi = 1 / phi

def generate_600cell_vertices():
    coords = []
    for signs in np.array(np.meshgrid(*[[-0.5,0.5]]*4)).T.reshape(-1,4):
        coords.append(signs)
    for i in range(4):
        for s in [-1,1]:
            v = np.zeros(4); v[i] = s; coords.append(v)
    base = [0.5*phi, 0.5, 0.5*inv_phi, 0]
    for perm in [[0,1,2,3],[0,1,3,2],[0,2,1,3],[0,2,3,1],[0,3,1,2],[0,3,2,1]]:
        for signs in [[1,1,1],[1,1,-1],[1,-1,1],[-1,1,1]]:
            v = np.zeros(4)
            v[perm[0]] = signs[0] * base[0]
            v[perm[1]] = signs[1] * base[1]
            v[perm[2]] = signs[2] * base[2]
            coords.append(v)
    verts = np.unique(np.round(coords, decimals=10), axis=0)
    verts = verts / np.linalg.norm(verts, axis=1)[:, None]
    return cp.asarray(verts, dtype=cp.float32)

vertices = generate_600cell_vertices()
N_VERT = vertices.shape[0]

dists = cp.linalg.norm(vertices[:, None] - vertices[None, :], axis=-1)
edge_threshold = cp.sort(dists.flatten())[720*2 + 1]
adj_mask = (dists < edge_threshold + 1e-6) & (dists > 1e-6)
neighbors = [cp.argwhere(adj_mask[i])[:,0] for i in range(N_VERT)]

print(f"Loaded: {N_VERT} vertices, avg degree ~{np.mean([len(n) for n in neighbors]):.1f}")

In [ ]:
# ====================== RUN ALL CONFIGS + REPEATS ======================
all_results = []
global_start = time.time()

for cfg in configs:
    print(f"\n=== Running config: {cfg['name']} ===")
    cfg_results = []

    for rep in range(1, N_REPEATS + 1):
        print(f"  Repeat {rep}/{N_REPEATS}")
        start = time.time()

        raw_amplitudes = cp.zeros(3, dtype=cp.complex64)

        for batch_idx in tqdm(range(n_batches), desc="Batches", leave=False):
            batch_size = min(BATCH_SIZE, N_paths_total - batch_idx * BATCH_SIZE)
            current = cp.random.randint(0, N_VERT, batch_size)
            phase = cp.zeros(batch_size, dtype=cp.complex64)
            fbs_grade = cp.ones(batch_size, dtype=cp.float32)

            for step in range(cfg['walk_length']):
                neigh_choices = cp.array([cp.random.choice(n, size=1)[0] for n in neighbors])[current]
                edge_len = cp.linalg.norm(vertices[current] - vertices[neigh_choices], axis=1)
                phase += 1j * cfg['phase_scale'] * edge_len * fbs_grade
                fbs_grade *= 1.0 / (edge_len ** cfg['fbs_power'] + 0.1)
                current = neigh_choices

            amp = cp.exp(phase) * fbs_grade

            # ──────────────── PATH-LENGTH WEIGHTING ────────────────
            weight = 1.0 / (step + 1) ** 1.5
            amp *= weight
            # ──────────────────────────────────────────────────────

            e_amp = amp * cp.exp(1j * 0.0)
            h_amp = amp * cp.exp(1j * np.pi / 2)
            q_amp = amp * cp.exp(1j * np.pi)

            raw_amplitudes[0] += cp.sum(e_amp)
            raw_amplitudes[1] += cp.sum(h_amp)
            raw_amplitudes[2] += cp.sum(q_amp)

            cp.cuda.Device().synchronize()  # flush GPU

        interference = raw_amplitudes / N_paths_total
        abs_interf = cp.abs(interference)
        total_mag = cp.sum(abs_interf)
        mean_amp = float(cp.mean(cp.abs(amp)))

        S = alpha / (2 * np.pi) * mean_amp
        mixing_sum = float(abs_interf[2] / total_mag) + 0.7 * float(abs_interf[1] / total_mag)
        delta_mu_e = C * mixing_sum * S

        result = {
            'config': cfg['name'],
            'repeat': rep,
            'delta_mu_e': delta_mu_e,
            'mean_amp': mean_amp,
            'S': S,
            'raw_sum_mag': float(cp.abs(raw_amplitudes).sum()),
            'time_sec': time.time() - start
        }
        cfg_results.append(result)
        all_results.append(result)

    # Per-config summary
    df_cfg = pd.DataFrame(cfg_results)
    print(f"\n{cfg['name']} summary (with weighting):")
    print(f"  Mean δμ_e:    {df_cfg['delta_mu_e'].mean():.3e}")
    print(f"  Median δμ_e:  {df_cfg['delta_mu_e'].median():.3e}")
    print(f"  Std δμ_e:     {df_cfg['delta_mu_e'].std():.3e}")
    print(f"  Best δμ_e:    {df_cfg['delta_mu_e'].min():.3e}")

df_all = pd.DataFrame(all_results)
df_all.to_csv("final_weighted_results.csv", index=False)
print(f"\nAll results saved to final_weighted_results.csv")
print(f"Total time: {time.time() - global_start:.1f} seconds")

## Summary Table
Run this after completion:

In [ ]:
df_all = pd.read_csv("final_weighted_results.csv")
print("Sorted by δμ_e (lowest first):")
print(df_all.sort_values('delta_mu_e').to_string(index=False))